# Do an Mon hoc: Hoc May (Machine Learning)
## De tai: Nhan dien khuon mat diem danh sinh vien bang cac thuat toan Hoc may truyen thong
---
**Giang vien:** TS. Ngo Quoc Viet
**Muc tieu:**
1. Thu thap va chuan hoa du lieu khuon mat (Data Preprocessing - ML-02).
2. Trich xuat dac trung va giam chieu du lieu bang PCA (Eigenfaces).
3. Huan luyen va danh gia so sanh cac thuat toan phan loai: k-NN, SVM, Decision Tree, Random Forest, AdaBoost, Naive Bayes (ML-04, ML-06, ML-07).
4. Truc quan hoa ma tran nham lan (Confusion Matrix) va phan tich chi so Accuracy, Precision, Recall, F1-Score.


In [ ]:
import sys
sys.path.append("../src")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from extract_features import prepare_data, load_face_dataset
from train_and_evaluate import get_candidate_models, evaluate_models
print("Libraries imported successfully.")


### 1. Kham pha va Truc quan hoa du lieu (Data Visualization)
Tai tap anh khuon mat sinh vien va hien thi mau anh truoc khi giam chieu du lieu.

In [ ]:
X, y, label_names = load_face_dataset("../dataset")
print(f"Tong so mau: {len(X)}, So chieu dac trung goc: {X.shape[1]}, So sinh vien: {len(label_names)}")

# Chon mau da dang tu cac sinh vien khac nhau
unique_classes = np.unique(y)
sample_indices = []
for c in unique_classes:
    idx_c = np.where(y == c)[0]
    sample_indices.extend(idx_c[:min(2, len(idx_c))])
sample_indices = sample_indices[:8] if len(sample_indices) >= 8 else np.linspace(0, len(X) - 1, min(8, len(X)), dtype=int)

fig, axes = plt.subplots(2, 4, figsize=(11, 5))
for ax, idx in zip(axes.flat, sample_indices):
    ax.imshow(X[idx].reshape(100, 100), cmap="gray")
    ax.set_title(label_names[y[idx]])
    ax.axis("off")
plt.suptitle("Mau anh khuon mat sinh vien (100x100 - Histogram Equalized)")
plt.tight_layout()
plt.show()


### 2. Trich xuat dac trung PCA (Eigenfaces) & Phan tich phuong sai giai thich
Su dung PCA de giam tu 10.000 chieu xuong khong gian n_components thanh phan chinh.

In [ ]:
from sklearn.decomposition import PCA
pca_full = PCA(n_components=min(50, len(X)), random_state=42)
pca_full.fit(X)

plt.figure(figsize=(8, 4))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_), marker="o", linestyle="--", color="b")
plt.xlabel("So luong thanh phan chinh (Number of Components)")
plt.ylabel("Tong phuong sai tich luy (Cumulative Explained Variance)")
plt.title("Do thi khuuy tay (Elbow Curve) cua PCA")
plt.grid(True)
plt.show()

fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    if i < len(pca_full.components_):
        ax.imshow(pca_full.components_[i].reshape(100, 100), cmap="bone")
        ax.set_title(f"Eigenface {i+1}")
    ax.axis("off")
plt.suptitle("Cac vector rieng Eigenfaces dai dien cho dac trung mat")
plt.show()


### 3. Huan luyen va Danh gia so sanh cac thuat toan phan loai
So sanh truc tiep: k-NN (k=3, k=5), SVM (Linear, RBF), Decision Tree, Random Forest, AdaBoost, Gaussian Naive Bayes, Logistic Regression.

In [ ]:
X_train_pca, X_test_pca, y_train, y_test, label_names, pca, *_ = prepare_data(
    dataset_dir="../dataset",
    test_size=0.25,
    pca_components=0.95,
    pca_save_path="../models/pca_model.pkl"
)

best_name, best_model, df_results = evaluate_models(
    X_train_pca, X_test_pca, y_train, y_test, label_names, output_dir="../reports"
)


### 4. Bieu do so sanh hieu nang cac thuat toan
Truc quan hoa F1-Score va Inference Time de phuc vu bao cao do an.

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=df_results, x="F1-Score", y="Algorithm", palette="viridis")
plt.title("So sanh F1-Score giua cac thuat toan Hoc May")
plt.xlim(0, 1.05)
plt.grid(axis="x", linestyle="--", alpha=0.7)
plt.show()
